In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame named 'forrest'
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')

# Display the first few rows of the DataFrame to verify the data has been loaded
print(train.head())

In [ ]:
train.info()

In [ ]:
#check if there is any class imbalancing
train['Cover_Type'].value_counts(normalize= True)

In [ ]:
# Find columns with all zero values
columns_with_all_zeros = train.columns[(train == 0).all()]

# Print or inspect the columns with all zero values
print("Columns with all zero values:", columns_with_all_zeros)


In [ ]:
train = train.drop(['Soil_Type7', 'Soil_Type15'], axis=1)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical features
numerical_features = train.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Create box plots
plt.figure(figsize=(16, 8))
sns.boxplot(data=train[numerical_features])
plt.title("Box Plots of Numerical Features")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
from scipy.stats import zscore

# Calculate z-scores for numerical features
z_scores = zscore(train[numerical_features])

# Define a threshold for outliers (e.g., 3 standard deviations)
threshold = 3

# Find rows with outliers
outlier_rows = (z_scores > threshold).any(axis=1)

# Display the number of rows before and after removing outliers
print("Number of rows before removing outliers:", len(train))
print("Number of rows after removing outliers:", len(train[~outlier_rows]))

# Remove outliers
train_no_outliers = train[~outlier_rows]


There are too many outliers. so we cannot remove them as we are losing lot of data rows.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import time

scaler = StandardScaler()

# Extract features and target variable
X = train.drop(['Id', 'Cover_Type'], axis=1)
y = train['Cover_Type']

In [ ]:
# Number of folds for cross-validation
num_folds = 5
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Initialize lists to store the results
accuracy_per_fold = []
loss_per_fold = []

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"Training on Fold {fold + 1}...")

    # Split the data into training and validation sets
    X_fold_train, X_fold_val = X.iloc[train_index], X.iloc[val_index]
    y_fold_train, y_fold_val = y.iloc[train_index], y.iloc[val_index]

    # Scale the data using the previously fitted scaler
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)  # Use transform, not fit_transform

    from keras.utils import to_categorical
    y_fold_train_encoded = to_categorical(y_fold_train - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6
    y_fold_val_encoded = to_categorical(y_fold_val - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6

    # Create and compile the model
    ann = Sequential()
    ann.add(Dense(units=128, activation='relu'))
    ann.add(Dense(units=64, activation='relu'))
    ann.add(Dense(units=7, activation='softmax'))
    ann.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Train the model on the training set for this fold
    start_time = time.time()
    history = ann.fit(X_fold_train_scaled, y_fold_train_encoded, batch_size=32, epochs=47, validation_data=(X_fold_val_scaled, y_fold_val_encoded))
    training_time = time.time()-start_time
    print("Training Time : {:.2f} seconds".format(training_time))
    # Evaluate the model on the validation set for this fold
    scores = ann.evaluate(X_fold_val_scaled, y_fold_val_encoded)
    loss_per_fold.append(scores[0])
    accuracy_per_fold.append(scores[1])

# Print the average performance across all folds
print("\nAverage Validation Loss: ", np.mean(loss_per_fold))
print("Average Validation Accuracy: ", np.mean(accuracy_per_fold))

# Plot training & validation accuracy values for the last fold
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

**Submission**

In [ ]:
import pandas as pd

# Load the test set into a DataFrame
test_set = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

# Display the first few rows of the test set
print(test_set.head())

In [ ]:
X_test = test_set.drop(['Id','Soil_Type7', 'Soil_Type15'], axis=1)

# Apply the same scaling to the test set features
X_test_scaled = scaler.transform(X_test.values)

start_time = time.time()
# Make predictions
predictions = ann.predict(X_test_scaled)
prediction_time = time.time()-start_time
print("prediction Time : {:.2f} seconds".format(prediction_time))

# Assuming you have a classification task and want to get the predicted class labels
predicted_labels = np.argmax(predictions, axis=1) + 1  # Add 1 to convert back to class labels starting from 1

# Create a DataFrame with the observation Id and predicted cover type
submission_df = pd.DataFrame({'Id': test_set['Id'], 'Cover_Type': predicted_labels})

print(submission_df)

# Save the DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)